# Think on the Graph: every output of the run MY29


## 0. Input
Program:
```java
int sum = 7;
int n = 2;
double avg = (double) (sum / n);
System.out.println(avg);
```
Question: What is printed? Explain what the (double) does here.

Student's answer: 3.5. The cast makes it a double division, so you get the decimal part
Settings: width 3, depth 3, prune llm, backend claude-cli, model claude-opus-5-5


## 1. Stage 1: the subgraph G from the code (no model)
Masked code (string and char literals and comments blanked; this is what the patterns see):
```
int sum = 7;
int n = 2;
double avg = (double) (sum / n);
System.out.println(avg);
```
Which code text tagged which concept (a generic concept fires on almost any Java line; it starts the walk only if the answer repeats its tag):
| concept | generic | channel | matched text | line |
|---|---|---|---|---|
| arithmetic_operators | False | pattern | 'm / n' | 3 |
| explicit_cast | False | pattern | '(double)' | 3 |
| expression | True | pattern | 'm /' | 3 |
| expression | True | pattern | '= 7;' | 1 |
| integer_division | False | signal | 'int sum = 7;\nint n = 2;\ndouble avg = (double) (sum / n' | 1 |
| primitive_type | True | pattern | 'int' | 1 |
| println | False | pattern | 'System.out.println(' | 4 |
| statement | True | pattern | ';' | 1 |
| type_conversion | False | pattern | '(double)' | 3 |

G: 8 concept nodes (3 generic), 6 concept edges, 39 misconception / error edges, 34 misconception / error nodes, 40 level nodes, 49 level-misconception links, 2 connected piece(s).
Note: concept nodes form 2 pieces (the KG has no edge between them): arithmetic_operators,explicit_cast,expression,integer_division,primitive_type,type_conversion | println,statement

Concept edges of G (the links the walk can read; `pre:` = prerequisite A -> B, `par:` = parallel):
| edge | type | rule (logic) |
|---|---|---|
| pre:arithmetic_operators->expression | prerequisite | An operator combines operand values a and b into a new value, so operators build larger expressions from smaller ones. |
| pre:arithmetic_operators->integer_division | prerequisite | For ints a and b, / gives the whole quotient q and % gives the remainder r, so that a = b * q + r. |
| pre:integer_division->explicit_cast | prerequisite | To divide two ints and keep the fraction, one operand is cast to double before the division, as in (double) freq[i] / T; casting the finished quotient comes too late. |
| pre:primitive_type->type_conversion | prerequisite | Conversion moves a value between primitive types: int to double happens automatically, while double to int needs a cast that drops the fraction. |
| pre:statement->println | prerequisite | System.out.println(s); is itself a statement, so its line of output appears at its own place in the sequence. |
| pre:type_conversion->explicit_cast | prerequisite | A cast is the explicit form of type conversion between primitive types, required when the conversion may lose information (e.g. (int) 2.71828 is 2). |

Misconception / error edges per concept (count), and level ways per concept (count):
| concept | misconception / error edges | level ways (L1-L4 + L0 non-misconception) |
|---|---|---|
| arithmetic_operators | 3 | 18 |
| explicit_cast | 6 | 19 |
| expression | 8 | 19 |
| integer_division | 6 | 16 |
| primitive_type | 2 | 21 |
| println | 6 | 17 |
| statement | 5 | 19 |
| type_conversion | 3 | 18 |

**Figure 1**

![Figure 1](figure01.png)
**Figure 2**

![Figure 2](figure02.png)


## 2. Clauses and start concepts (no model)
The answer split into clauses (at sentence ends, ';', and before 'because' / 'so'; quotes removed):
| clause | text |
|---|---|
| c1 | 3.5 |
| c2 | The cast makes it a double division |
| c3 | so you get the decimal part |

start: BM25 of the answer against each tagged concept: explicit_cast 7.80 (hits 1), integer_division 6.80 (hits 1), type_conversion 3.01 (hits 1), println 2.85 (hits 1), arithmetic_operators 2.45 (hits 1)

start: chosen explicit_cast, integer_division, type_conversion

start concepts (3): explicit_cast, integer_division, type_conversion

Start concepts: explicit_cast, integer_division, type_conversion

**Figure 3**

![Figure 3](figure03.png)


## 3. The walk, depth by depth
At each depth, for each current concept: the edges offered (a concept entering as a start concept offers its misconception / error edges and its level ways, as two families; otherwise it offers its concept edges), the prune prompt and the model's reply, then the label prompt(s) and the model's reply, parsed into label / clauses / confidence. Score = prune score x confidence.

### Depth 1: current concepts 'explicit_cast', 'integer_division', 'type_conversion'

#### explicit_cast, misconceptions: 6 edges offered
Edges offered:
| edge |
|---|
| err:explicit_cast->RoundReturnsInt \| wrong rule: Writes int n = Math.round(x); because Math.round gives an int. \| right rule: Math.round(double) returns a long, so storing it in an int needs a cast: int n = (int) Math.round(x); |
| mis:explicit_cast->CastAfterIntDivision \| wrong rule: (double) (3 / 5) is 0.6. \| right rule: The integer division 3 / 5 = 0 happens before the cast, so the result is 0.0. |
| mis:explicit_cast->CastAppliesToWholeExpression \| wrong rule: (int) 2.5 / 0.15 casts the whole quotient to int. \| right rule: The cast applies only to what comes right after it, here 2.5. |
| mis:explicit_cast->CastParsesString \| wrong rule: (int) args[0] converts the text "34" into the number 34. \| right rule: A String argument that represents a number must be converted with a method such as Integer.parseInt. |
| mis:explicit_cast->CastRounds \| wrong rule: (int) 2.7 evaluates to 3. \| right rule: A cast to int discards the fractional part (rounds toward zero, so (int) -2.7 is -2); Math.round rounds to the nearest integer. |
| mis:explicit_cast->LaterDoubleLiteralFixesDivision \| wrong rule: Changing a later literal to 100.0 fixes an earlier int division. \| right rule: The int division happens first; the double must be in the division itself. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:explicit_cast->CastAfterIntDivision (0.70), mis:explicit_cast->LaterDoubleLiteralFixesDivision (0.20), mis:explicit_cast->CastAppliesToWholeExpression (0.10)
```
Kept by pruning: mis:explicit_cast->CastAfterIntDivision (0.70), mis:explicit_cast->LaterDoubleLiteralFixesDivision (0.20), mis:explicit_cast->CastAppliesToWholeExpression (0.10)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:explicit_cast->CastAfterIntDivision \| wrong rule: (double) (3 / 5) is 0.6. \| right rule: The integer division 3 / 5 = 0 happens before the cast, so the result is 0.0. |
| mis:explicit_cast->LaterDoubleLiteralFixesDivision \| wrong rule: Changing a later literal to 100.0 fixes an earlier int division. \| right rule: The int division happens first; the double must be in the division itself. |
| mis:explicit_cast->CastAppliesToWholeExpression \| wrong rule: (int) 2.5 / 0.15 casts the whole quotient to int. \| right rule: The cast applies only to what comes right after it, here 2.5. |
Model's reply:
```
mis:explicit_cast->CastAfterIntDivision | held | clauses: 1, 2, 3 | confidence: 0.95
mis:explicit_cast->LaterDoubleLiteralFixesDivision | not applied | clauses: none | confidence: 0.9
mis:explicit_cast->CastAppliesToWholeExpression | not applied | clauses: none | confidence: 0.8
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:explicit_cast->CastAfterIntDivision | held | 0.7000000000000001 | 0.95 | 0.67 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |
| mis:explicit_cast->LaterDoubleLiteralFixesDivision | not applied |  |  | 0.0 |  |  |
| mis:explicit_cast->CastAppliesToWholeExpression | not applied |  |  | 0.0 |  |  |

#### explicit_cast, level ways: 19 edges offered
Edges offered:
| edge |
|---|
| lvl:explicit_cast@L0#0 \| level L0 way everyday_reading: Folk meaning: 'convert 2.7 to a whole number' means round it to 3, and (int) looks like a function label that applies to everything written after it. Java rule: a cast to int truncates toward zero ((int) 2.7 is 2) and, having higher precedence than arithmetic, applies only to the value immediately following it. |
| lvl:explicit_cast@L0#4 \| level L0 way inconsistent_trace: Points to LT (the cast is a unary operator applied to the operand right after it, before * and /). Applied unevenly: in one line the student casts only the first operand, and in another applies the same cast to the whole expression. (here mixing in: "(int) 2.7 evaluates to 3") |
| lvl:explicit_cast@L1#0 \| level L1 way names_the_element: A cast is a unary operator, written (type) expression, that converts a value of one primitive type to another, used when the conversion may lose information. |
| lvl:explicit_cast@L1#1 \| level L1 way states_the_rule: (T) e converts the value of e to type T; a double-to-int cast discards the fraction by rounding toward zero; a cast is required for narrowing conversions; it has higher precedence than arithmetic; casts between boolean and numbers, or from String to int, are not allowed. |
| lvl:explicit_cast@L2#0 \| level L2 way line_by_line_trace: Evaluate the operand immediately to the right of the cast (a parenthesized expression is evaluated completely first), convert that value, then continue with the remaining operators of the expression. |
| lvl:explicit_cast@L2#1 \| level L2 way output_prediction: (int) 2.71828 is int 2; (int) -2.7 is -2; (int) 11 * 0.3 is double 3.3; (int) (11 * 0.3) is int 3; (double) 3 / 5 is 0.6 but (double) (3 / 5) is 0.0; (char) ('a' + 4) is 'e'. |
| lvl:explicit_cast@L2#2 \| level L2 way one_step_execution: A cast yields a new value and changes nothing else: neither the type nor the value of the variable being cast changes (after int x = (int) pi; pi is still 3.14159). |
| lvl:explicit_cast@L3#0 \| level L3 way relation_to_neighbour: A cast is the explicit conversion between primitive types, needed exactly where automatic conversion is refused because information may be lost. |
| lvl:explicit_cast@L3#1 \| level L3 way relation_to_neighbour: To turn Math.random()'s double into a random int from 0 to N-1, the cast must enclose the product: (int) (Math.random() * N); casting first gives 0. |
| lvl:explicit_cast@L3#2 \| level L3 way relation_to_neighbour: x op= e contains an implicit cast back to the type of x, so with int x, x += 0.5 compiles (x = (int) (x + 0.5)) while x = x + 0.5 is a compile-time error. |
| lvl:explicit_cast@L3#3 \| level L3 way relation_to_neighbour: Widening (promotion) happens automatically when a value goes to a type with a larger range (int to double) and loses nothing; a narrowing conversion (double to int) is never automatic and needs an explicit cast, which truncates toward zero. |
| lvl:explicit_cast@L3#4 \| level L3 way relation_to_neighbour: (int) 2.71828 truncates toward zero and gives 2; Math.round(2.71828) rounds to the nearest integer and gives 3, as a long that needs (int) to become an int. |
| lvl:explicit_cast@L3#5 \| level L3 way purpose_of_the_whole: Used to discard information on purpose: truncating a scaled random number to an index, taking the integer part, converting between char and int codes ((char) ('a' + k)), or forcing real division with (double) sum / n. |
| lvl:explicit_cast@L4#0 \| level L4 way predict_a_change: Moving the parentheses changes when truncation happens ((int) (a * b) vs (int) a * b); removing a cast from a narrowing assignment turns it into a compile-time error; int to double to int restores the value, but double to int to double loses the fraction. |
| lvl:explicit_cast@L4#1 \| level L4 way transfer_to_a_new_program: Use a cast when you intend to truncate or narrow (random indices, integer parts, char codes); use Math.round to round, Integer.parseInt for Strings, and put a double into the division itself rather than casting its int result. |
| lvl:explicit_cast@L4#2 \| level L4 way diagnose_an_error: rule broken: Math.round(double) returns a long, so storing it in an int needs a cast: int n = (int) Math.round(x);. Kind: compile-time. Trigger class: int n = Math.round(x); with double x. |
| lvl:explicit_cast@L4#3 \| level L4 way diagnose_an_error: rule broken: A cast applies only to the value right after it. Kind: compile-time. Trigger class: int n = (int) 11 * 0.3; the product is still a double (possible loss of precision); in a double context the same mistake is a silent logic error ((int) pi * 20.0 gives 60.0). |
| lvl:explicit_cast@L4#4 \| level L4 way diagnose_an_error: rule broken: Casts require compatible types. Kind: compile-time. Trigger class: (int) "3" or (int) args[0] on a String, or a cast between boolean and a number. |
| lvl:explicit_cast@L4#5 \| level L4 way diagnose_an_error: rule broken: A narrowing cast may lose magnitude, not just the fraction. Kind: silent logic error. Trigger class: Casting a value outside the target range, e.g. (int) of a long above 2147483647 or of a huge double. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:explicit_cast@L2#1 (Score: 0.4)}
{lvl:explicit_cast@L0#0 (Score: 0.35)}
{lvl:explicit_cast@L2#0 (Score: 0.25)}
```
Kept by pruning: lvl:explicit_cast@L2#1 (0.40), lvl:explicit_cast@L0#0 (0.35), lvl:explicit_cast@L2#0 (0.25)

Label prompt for level way lvl:explicit_cast@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> (int) 2.71828 is int 2; (int) -2.7 is -2; (int) 11 * 0.3 is double 3.3; (int) (11 * 0.3) is int 3; (double) 3 / 5 is 0.6 but (double) (3 / 5) is 0.0; (char) ('a' + 4) is 'e'.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.9
```

Label prompt for level way lvl:explicit_cast@L0#0 (the model saw only the concept, the way's name 'everyday reading' and its text; no level):
> Folk meaning: 'convert 2.7 to a whole number' means round it to 3, and (int) looks like a function label that applies to everything written after it. Java rule: a cast to int truncates toward zero ((int) 2.7 is 2) and, having higher precedence than arithmetic, applies only to the value immediately following it.

Model's reply:
```
Label: shown
Clauses: 1, 2
Confidence: 0.6
```

Label prompt for level way lvl:explicit_cast@L2#0 (the model saw only the concept, the way's name 'line by line trace' and its text; no level):
> Evaluate the operand immediately to the right of the cast (a parenthesized expression is evaluated completely first), convert that value, then continue with the remaining operators of the expression.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:explicit_cast@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:explicit_cast@L0#0 | shown | 0.35 | 0.6 | 0.21 | c1, c2 | 3.5; The cast makes it a double division |
| lvl:explicit_cast@L2#0 | not applied |  |  | 0.0 |  |  |

#### integer_division, misconceptions: 6 edges offered
Edges offered:
| edge |
|---|
| mis:integer_division->AllDoublesToAvoidDivision \| wrong rule: Making every variable a double is the right fix for integer division. \| right rule: Use int for countable things and double for measured ones; where a fraction is needed, make one operand of the division a double (or cast) instead of changing every type. |
| mis:integer_division->CastAfterIntDivision \| wrong rule: (double) (3 / 5) is 0.6. \| right rule: The integer division 3 / 5 = 0 happens before the cast, so the result is 0.0. |
| mis:integer_division->IntegerDivisionGivesFraction \| wrong rule: 10 / 3 evaluates to 3.333.... \| right rule: Dividing two ints gives an int: the fractional part is discarded. |
| mis:integer_division->LaterDoubleLiteralFixesDivision \| wrong rule: Changing a later literal to 100.0 fixes an earlier int division. \| right rule: The int division happens first; the double must be in the division itself. |
| mis:integer_division->NegativeRemainderSign \| wrong rule: a % b is never negative, and integer division of a negative number rounds down. \| right rule: a / b rounds toward 0 and a % b takes the sign of a: -14 / 3 is -4 and -14 % 3 is -2. |
| mis:integer_division->TargetTyping \| wrong rule: The type of a numerical expression depends on the type expected by the surrounding context, so double d = a / b; keeps the fraction. \| right rule: The type of a numerical expression is determined by the expression, so double d = a / b; with int a and b still divides as ints. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:integer_division->CastAfterIntDivision (0.60), mis:integer_division->TargetTyping (0.25), mis:integer_division->IntegerDivisionGivesFraction (0.15)
```
Kept by pruning: mis:integer_division->CastAfterIntDivision (0.60), mis:integer_division->TargetTyping (0.25), mis:integer_division->IntegerDivisionGivesFraction (0.15)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:integer_division->CastAfterIntDivision \| wrong rule: (double) (3 / 5) is 0.6. \| right rule: The integer division 3 / 5 = 0 happens before the cast, so the result is 0.0. |
| mis:integer_division->TargetTyping \| wrong rule: The type of a numerical expression depends on the type expected by the surrounding context, so double d = a / b; keeps the fraction. \| right rule: The type of a numerical expression is determined by the expression, so double d = a / b; with int a and b still divides as ints. |
| mis:integer_division->IntegerDivisionGivesFraction \| wrong rule: 10 / 3 evaluates to 3.333.... \| right rule: Dividing two ints gives an int: the fractional part is discarded. |
Model's reply:
```
mis:integer_division->CastAfterIntDivision | held | clauses: 1, 2, 3 | confidence: 0.95
mis:integer_division->TargetTyping | not applied | clauses: none | confidence: 0.8
mis:integer_division->IntegerDivisionGivesFraction | not applied | clauses: none | confidence: 0.75
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:integer_division->CastAfterIntDivision | held | 0.6 | 0.95 | 0.57 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |
| mis:integer_division->TargetTyping | not applied |  |  | 0.0 |  |  |
| mis:integer_division->IntegerDivisionGivesFraction | not applied |  |  | 0.0 |  |  |

#### integer_division, level ways: 16 edges offered
Edges offered:
| edge |
|---|
| lvl:integer_division@L0#0 \| level L0 way everyday_reading: In everyday and school arithmetic, 'divide' gives an exact answer or a fraction (10 ÷ 3 = 3 1/3), and rounding goes to the nearest whole number. In Java, / with two int operands discards the fraction, rounding toward zero (59 / 60 is 0). |
| lvl:integer_division@L0#4 \| level L0 way inconsistent_trace: Points to LT (each / looks at its own two operand types when it is evaluated). Applied unevenly, a student evaluates 9/5 as 1.8 in f = (9/5)*c + 32 but as 1 elsewhere, or assumes a later double operand rescues an earlier int division. (here mixing in: "The type of a numerical expression depends on the type expected by the surrounding context, so double d = a / b; keeps the fraction") |
| lvl:integer_division@L1#0 \| level L1 way names_the_element: Integer division: / applied to two integer operands, giving the whole-number quotient with the fractional part discarded. |
| lvl:integer_division@L1#1 \| level L1 way states_the_rule: If both operands of / are integral types, the result is an integer that rounds toward 0 (-14 / 3 is -4). If either operand is a double, the division is floating point. |
| lvl:integer_division@L2#0 \| level L2 way line_by_line_trace: Each / is evaluated when its turn comes (precedence, then left to right), using the types of its own two operands. Promoting the result to double, by a cast or by assignment, happens only afterwards and cannot restore the lost fraction. |
| lvl:integer_division@L2#1 \| level L2 way output_prediction: 10 / 3 is 3, 59 / 60 is 0, 1234 / 99 is 12, -14 / 3 is -4, and (double) (3 / 5) is 0.0. |
| lvl:integer_division@L2#2 \| level L2 way one_step_execution: q = a / b stores the truncated quotient in q and leaves a and b unchanged. If b is 0, an ArithmeticException stops execution before anything is stored. |
| lvl:integer_division@L3#0 \| level L3 way relation_to_neighbour: / and % on ints are paired: q = a / b and r = a % b always satisfy a = b * q + r. |
| lvl:integer_division@L3#1 \| level L3 way relation_to_neighbour: Normalizing counts must divide as doubles. With two ints, counts / total truncates to 0 whenever counts < total. |
| lvl:integer_division@L3#2 \| level L3 way relation_to_neighbour: % gives the remainder of the same truncating division: (a / b) * b + a % b == a. |
| lvl:integer_division@L3#3 \| level L3 way purpose_of_the_whole: Quotient plans: unit conversion (76 / 12 feet), dropping a digit (n / 10), finding a midpoint ((lo + hi) / 2), and computing a bucket or row index (i / cols). |
| lvl:integer_division@L4#0 \| level L4 way predict_a_change: Making one operand a double (10.0 / 3) keeps the fraction (3.333...). Casting after the division does not. Multiplying before dividing preserves precision (59 * 100 / 60 is 98). Changing an operand's sign changes only the quotient's sign, since rounding is toward zero. |
| lvl:integer_division@L4#1 \| level L4 way transfer_to_a_new_program: Use integer division when you want the whole quotient (units, digits, indices, half a size). When a fraction is needed, make one operand a double (10.0 / 3 or (double) a / b) rather than turning every variable into a double. |
| lvl:integer_division@L4#2 \| level L4 way diagnose_an_error: rule broken: With two int operands, / discards the fraction. Kind: silent logic error. Trigger class: Two int operands where a fraction is intended: 9/5 in a temperature formula, count / total, 1 / n, sum / n for an average. |
| lvl:integer_division@L4#3 \| level L4 way diagnose_an_error: rule broken: int division by zero is undefined. Kind: run-time exception. Trigger class: A divisor that evaluates to 0: ArithmeticException: / by zero. |
| lvl:integer_division@L4#4 \| level L4 way diagnose_an_error: rule broken: Integer division rounds toward zero, not down. Kind: silent logic error. Trigger class: Negative operands where floor was expected (-7 / 2 is -3, not -4). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:integer_division@L0#4 (Score: 0.6)}
{lvl:integer_division@L2#0 (Score: 0.25)}
{lvl:integer_division@L4#0 (Score: 0.15)}
```
Kept by pruning: lvl:integer_division@L0#4 (0.60), lvl:integer_division@L2#0 (0.25), lvl:integer_division@L4#0 (0.15)

Label prompt for level way lvl:integer_division@L0#4 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Points to LT (each / looks at its own two operand types when it is evaluated). Applied unevenly, a student evaluates 9/5 as 1.8 in f = (9/5)*c + 32 but as 1 elsewhere, or assumes a later double operand rescues an earlier int division. (here mixing in: "The type of a numerical expression depends on the type expected by the surrounding context, so double d = a / b; keeps the fraction")

Model's reply:
```
Label: shown
Clauses: 1, 2, 3
Confidence: 0.85
```

Label prompt for level way lvl:integer_division@L2#0 (the model saw only the concept, the way's name 'line by line trace' and its text; no level):
> Each / is evaluated when its turn comes (precedence, then left to right), using the types of its own two operands. Promoting the result to double, by a cast or by assignment, happens only afterwards and cannot restore the lost fraction.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:integer_division@L4#0 (the model saw only the concept, the way's name 'predict a change' and its text; no level):
> Making one operand a double (10.0 / 3) keeps the fraction (3.333...). Casting after the division does not. Multiplying before dividing preserves precision (59 * 100 / 60 is 98). Changing an operand's sign changes only the quotient's sign, since rounding is toward zero.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.92
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:integer_division@L0#4 | shown | 0.6 | 0.85 | 0.51 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |
| lvl:integer_division@L2#0 | not applied |  |  | 0.0 |  |  |
| lvl:integer_division@L4#0 | not applied |  |  | 0.0 |  |  |

#### type_conversion, misconceptions: 3 edges offered
Edges offered:
| edge |
|---|
| mis:type_conversion->NarrowingAssignmentAllowed \| wrong rule: A double value can be stored in an int variable like an int in a double. \| right rule: int to double is automatic; double to int needs a cast. |
| mis:type_conversion->NoCastIfSameSize \| wrong rule: If a variable is at least as big as a value, no cast is needed to assign the value. \| right rule: Assigning a double to an int needs a cast even when the value would fit, because a conversion that might involve loss of information leads to a compile-time error. |
| mis:type_conversion->TargetTyping \| wrong rule: The type of a numerical expression depends on the type expected by the surrounding context, so double d = a / b; keeps the fraction. \| right rule: The type of a numerical expression is determined by the expression, so double d = a / b; with int a and b still divides as ints. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:type_conversion->TargetTyping (0.80), mis:type_conversion->NarrowingAssignmentAllowed (0.10), mis:type_conversion->NoCastIfSameSize (0.10)
```
Kept by pruning: mis:type_conversion->TargetTyping (0.80), mis:type_conversion->NarrowingAssignmentAllowed (0.10), mis:type_conversion->NoCastIfSameSize (0.10)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:type_conversion->TargetTyping \| wrong rule: The type of a numerical expression depends on the type expected by the surrounding context, so double d = a / b; keeps the fraction. \| right rule: The type of a numerical expression is determined by the expression, so double d = a / b; with int a and b still divides as ints. |
| mis:type_conversion->NarrowingAssignmentAllowed \| wrong rule: A double value can be stored in an int variable like an int in a double. \| right rule: int to double is automatic; double to int needs a cast. |
| mis:type_conversion->NoCastIfSameSize \| wrong rule: If a variable is at least as big as a value, no cast is needed to assign the value. \| right rule: Assigning a double to an int needs a cast even when the value would fit, because a conversion that might involve loss of information leads to a compile-time error. |
Model's reply:
```
mis:type_conversion->TargetTyping | held | clauses: 1, 2, 3 | confidence: 0.8
mis:type_conversion->NarrowingAssignmentAllowed | not applied | clauses: none | confidence: 0.9
mis:type_conversion->NoCastIfSameSize | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:type_conversion->TargetTyping | held | 0.8 | 0.8 | 0.64 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |
| mis:type_conversion->NarrowingAssignmentAllowed | not applied |  |  | 0.0 |  |  |
| mis:type_conversion->NoCastIfSameSize | not applied |  |  | 0.0 |  |  |

#### type_conversion, level ways: 18 edges offered
Edges offered:
| edge |
|---|
| lvl:type_conversion@L0#0 \| level L0 way everyday_reading: Folk meaning: a number is a number, so 3, 3.0 and "3" are the same value, and a variable will take whatever form the context needs (the result 'becomes' double when stored in a double). Java rule: every value has a fixed type; conversions happen only by automatic promotion to a wider type, by an explicit cast, by conversion methods (Integer.parseInt, Math.round), or by String conversion with +. |
| lvl:type_conversion@L0#3 \| level L0 way inconsistent_trace: Points to LT (each subexpression is computed in its own type, and conversion happens only where the types meet). Applied unevenly: the student promotes before an int division on one line (treating 9/5 as 1.8) but not on another, or casts the whole expression on one pass and only the first operand on the next. (here mixing in: "If a variable is at least as big as a value, no cast is needed to assign the value") |
| lvl:type_conversion@L1#0 \| level L1 way names_the_element: Type conversion is changing a value of one type into a value of another type: automatic promotion (widening), an explicit cast, an explicit conversion method (Integer.parseInt, Double.parseDouble, Math.round), or automatic conversion to String. |
| lvl:type_conversion@L1#1 \| level L1 way states_the_rule: Java converts automatically to a type with a larger range (promotion) because no information is lost; a conversion that might lose information (double to int) requires an explicit cast or is a compile-time error; Strings and numbers convert only via methods or string conversion. |
| lvl:type_conversion@L2#0 \| level L2 way line_by_line_trace: Innermost subexpressions are evaluated in their own types first; at each binary operator the narrower operand is promoted; a cast converts the single value right after it; at an assignment the final value is converted to the variable's type (widening allowed, narrowing rejected at compile time). |
| lvl:type_conversion@L2#1 \| level L2 way output_prediction: Typical results: "1234" + 99 is the String "123499"; Integer.parseInt("123") is int 123; (int) 2.71828 is int 2; Math.round(2.71828) is long 3; 11 * 0.3 is double 3.3; 11 * (int) 0.3 is int 0; (int) (11 * 0.3) is int 3. |
| lvl:type_conversion@L2#2 \| level L2 way one_step_execution: A conversion produces a new value and never changes the source variable or its declared type: after int x = (int) pi; x is 3 and pi still holds 3.14159. |
| lvl:type_conversion@L3#0 \| level L3 way relation_to_neighbour: Conversions between primitive types follow their ranges: int to double happens automatically, double to int needs a cast that drops the fraction, and boolean converts to no numeric type. |
| lvl:type_conversion@L3#1 \| level L3 way relation_to_neighbour: Parsing methods are explicit type conversion from String to a number: a method takes an argument of one type and produces a result of another. |
| lvl:type_conversion@L3#2 \| level L3 way relation_to_neighbour: A cast is the explicit conversion between primitive types, required when information may be lost: (int) 2.71828 is 2. |
| lvl:type_conversion@L3#3 \| level L3 way relation_to_neighbour: Widening (promotion) is the automatic kind of conversion, applied at operators and assignments when the target type has a larger range. |
| lvl:type_conversion@L3#4 \| level L3 way purpose_of_the_whole: Adapts data between parts of a program: command-line text to numbers, int counts to double for an average ((double) sum / n), doubles to int indices or counts. |
| lvl:type_conversion@L4#0 \| level L4 way predict_a_change: Moving the point of conversion changes the result: (double) (a / b) differs from (double) a / b, and (int) 11 * 0.3 differs from (int) (11 * 0.3). Widening int to double and back is reversible for int values; narrowing double to int loses the fraction irreversibly. |
| lvl:type_conversion@L4#1 \| level L4 way transfer_to_a_new_program: Rely on automatic promotion when mixing int with double; write a cast only when you intend to drop information; use parse methods for text input and string conversion for output; paying attention to types avoids subtle bugs. |
| lvl:type_conversion@L4#2 \| level L4 way diagnose_an_error: rule broken: A conversion that may lose information needs an explicit cast. Kind: compile-time. Trigger class: Assigning or passing a double (or long) expression to an int, e.g. int n = 11 * 0.3; (possible loss of precision). |
| lvl:type_conversion@L4#3 \| level L4 way diagnose_an_error: rule broken: Promotion happens per operator, after an int division has already truncated. Kind: silent logic error. Trigger class: All-int division inside a double formula, e.g. double y = 1 / 3; or (9/5) * c + 32. |
| lvl:type_conversion@L4#4 \| level L4 way diagnose_an_error: rule broken: Casts work only between compatible types; a String is converted with a parse method. Kind: compile-time. Trigger class: (int) str with a String str, or (int) flag with a boolean. |
| lvl:type_conversion@L4#5 \| level L4 way diagnose_an_error: rule broken: A parse method accepts only a valid textual representation of the number. Kind: run-time exception. Trigger class: Integer.parseInt of text that is not an integer in digits ("five", "12.2", ""). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:type_conversion@L4#3 (Score: 0.4)}
{lvl:type_conversion@L4#0 (Score: 0.35)}
{lvl:type_conversion@L0#0 (Score: 0.25)}
```
Kept by pruning: lvl:type_conversion@L4#3 (0.40), lvl:type_conversion@L4#0 (0.35), lvl:type_conversion@L0#0 (0.25)

Label prompt for level way lvl:type_conversion@L4#3 (the model saw only the concept, the way's name 'diagnose an error' and its text; no level):
> rule broken: Promotion happens per operator, after an int division has already truncated. Kind: silent logic error. Trigger class: All-int division inside a double formula, e.g. double y = 1 / 3; or (9/5) * c + 32.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:type_conversion@L4#0 (the model saw only the concept, the way's name 'predict a change' and its text; no level):
> Moving the point of conversion changes the result: (double) (a / b) differs from (double) a / b, and (int) 11 * 0.3 differs from (int) (11 * 0.3). Widening int to double and back is reversible for int values; narrowing double to int loses the fraction irreversibly.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.93
```

Label prompt for level way lvl:type_conversion@L0#0 (the model saw only the concept, the way's name 'everyday reading' and its text; no level):
> Folk meaning: a number is a number, so 3, 3.0 and "3" are the same value, and a variable will take whatever form the context needs (the result 'becomes' double when stored in a double). Java rule: every value has a fixed type; conversions happen only by automatic promotion to a wider type, by an explicit cast, by conversion methods (Integer.parseInt, Math.round), or by String conversion with +.

Model's reply:
```
Label: shown
Clauses: 1, 2, 3
Confidence: 0.72
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:type_conversion@L4#3 | not applied |  |  | 0.0 |  |  |
| lvl:type_conversion@L4#0 | not applied |  |  | 0.0 |  |  |
| lvl:type_conversion@L0#0 | shown | 0.25 | 0.72 | 0.18 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |

**Kept on a path at this depth** (best N in tiers: held misconception / broken edge, then held error, then correct / confirmed / shown):

mis:explicit_cast->CastAfterIntDivision [held, tier 0, 0.67], mis:type_conversion->TargetTyping [held, tier 0, 0.64], mis:integer_division->CastAfterIntDivision [held, tier 0, 0.57]

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | 3.5 | mis:explicit_cast->CastAfterIntDivision |
| c2 | The cast makes it a double division | mis:explicit_cast->CastAfterIntDivision |
| c3 | so you get the decimal part | mis:explicit_cast->CastAfterIntDivision |

**Graph check, not yet read (the walk may not stop while these remain):** pre:arithmetic_operators->integer_division, pre:integer_division->explicit_cast, pre:primitive_type->type_conversion, pre:type_conversion->explicit_cast

**Figure 4 (the graph after depth 1)**

![Figure 4](figure04.png)
### Depth 2: current concepts 'explicit_cast', 'type_conversion', 'integer_division'

#### explicit_cast, concept edges: 2 edges offered
Edges offered:
| edge |
|---|
| pre:integer_division->explicit_cast \| rule: To divide two ints and keep the fraction, one operand is cast to double before the division, as in (double) freq[i] / T; casting the finished quotient comes too late. |
| pre:type_conversion->explicit_cast \| rule: A cast is the explicit form of type conversion between primitive types, required when the conversion may lose information (e.g. (int) 2.71828 is 2). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:integer_division->explicit_cast (Score: 0.8)}
{pre:type_conversion->explicit_cast (Score: 0.2)}
```
Kept by pruning: pre:integer_division->explicit_cast (0.80), pre:type_conversion->explicit_cast (0.20)

Label prompt for concept edge pre:integer_division->explicit_cast (integer_division -> explicit_cast, prerequisite):
> To divide two ints and keep the fraction, one operand is cast to double before the division, as in (double) freq[i] / T; casting the finished quotient comes too late.

Model's reply:
```
Label: broken
Clauses: 1, 2, 3
Confidence: 0.95
```

Label prompt for concept edge pre:type_conversion->explicit_cast (type_conversion -> explicit_cast, prerequisite):
> A cast is the explicit form of type conversion between primitive types, required when the conversion may lose information (e.g. (int) 2.71828 is 2).

Model's reply:
```
Label: broken
Clauses: 1, 2, 3
Confidence: 0.75
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:integer_division->explicit_cast | broken | 0.8 | 0.95 | 0.76 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |
| pre:type_conversion->explicit_cast | broken | 0.2 | 0.75 | 0.15 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |

#### type_conversion, concept edges: 1 edges offered
Edges offered:
| edge |
|---|
| pre:primitive_type->type_conversion \| rule: Conversion moves a value between primitive types: int to double happens automatically, while double to int needs a cast that drops the fraction. |
| pre:type_conversion->explicit_cast \| rule: A cast is the explicit form of type conversion between primitive types, required when the conversion may lose information (e.g. (int) 2.71828 is 2). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:primitive_type->type_conversion (Score: 1.0)}
```
Kept by pruning: pre:primitive_type->type_conversion (1.00)

Label prompt for concept edge pre:primitive_type->type_conversion (primitive_type -> type_conversion, prerequisite):
> Conversion moves a value between primitive types: int to double happens automatically, while double to int needs a cast that drops the fraction.

Model's reply:
```
Label: broken
Clauses: 1, 2, 3
Confidence: 0.7
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:primitive_type->type_conversion | broken | 1.0 | 0.7 | 0.7 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |

#### integer_division, concept edges: 1 edges offered
Edges offered:
| edge |
|---|
| pre:arithmetic_operators->integer_division \| rule: For ints a and b, / gives the whole quotient q and % gives the remainder r, so that a = b * q + r. |
| pre:integer_division->explicit_cast \| rule: To divide two ints and keep the fraction, one operand is cast to double before the division, as in (double) freq[i] / T; casting the finished quotient comes too late. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:arithmetic_operators->integer_division (Score: 1.0)}
```
Kept by pruning: pre:arithmetic_operators->integer_division (1.00)

Label prompt for concept edge pre:arithmetic_operators->integer_division (arithmetic_operators -> integer_division, prerequisite):
> For ints a and b, / gives the whole quotient q and % gives the remainder r, so that a = b * q + r.

Model's reply:
```
Label: broken
Clauses: 1, 2, 3
Confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:arithmetic_operators->integer_division | broken | 1.0 | 0.9 | 0.9 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |

**Kept on a path at this depth** (best N in tiers: held misconception / broken edge, then held error, then correct / confirmed / shown):

pre:arithmetic_operators->integer_division [broken, tier 0, 0.90], pre:integer_division->explicit_cast [broken, tier 0, 0.76], pre:primitive_type->type_conversion [broken, tier 0, 0.70], pre:type_conversion->explicit_cast [broken, tier 0, 0.15]

reached across pre:arithmetic_operators->integer_division, entered next: arithmetic_operators

reached across pre:primitive_type->type_conversion, entered next: primitive_type

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | 3.5 | mis:explicit_cast->CastAfterIntDivision |
| c2 | The cast makes it a double division | mis:explicit_cast->CastAfterIntDivision |
| c3 | so you get the decimal part | mis:explicit_cast->CastAfterIntDivision |

**Graph check, not yet read (the walk may not stop while these remain):** own edges of arithmetic_operators, own edges of primitive_type

**Figure 5 (the graph after depth 2)**

![Figure 5](figure05.png)
### Depth 3: current concepts 'arithmetic_operators', 'integer_division', 'explicit_cast', 'primitive_type', 'type_conversion'

#### arithmetic_operators, misconceptions: 3 edges offered
Edges offered:
| edge |
|---|
| mis:arithmetic_operators->ArithmeticOnNumericStrings \| wrong rule: "10" - 1 or "6" * "2" works because the strings look like numbers. \| right rule: Arithmetic operators other than + are not defined for Strings; a String of digits must first be converted to a number, e.g. with Integer.parseInt. |
| mis:arithmetic_operators->CaretIsPower \| wrong rule: a^2 computes a squared. \| right rule: Java has no exponent operator; ^ is exclusive or, and a * a or Math.pow(a, 2) computes a squared. |
| mis:arithmetic_operators->RemainderIsSubtraction \| wrong rule: 9 % 5 means 9 minus 5 (a % b subtracts b from a). \| right rule: % gives the remainder after integer division: 9 % 5 is 4 because 9 = 1 * 5 + 4, and 17 % 5 is 2, not 12. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```

```
Kept by pruning: 

Parsed labels:
(none)

#### arithmetic_operators, level ways: 18 edges offered
Edges offered:
| edge |
|---|
| lvl:arithmetic_operators@L0#0 \| level L0 way everyday_reading: In maths notation, multiplication can be implicit (2x), ^ or a superscript means a power, ÷ is exact division, and % means percent. In Java * must be written, / depends on the operand types, % is the remainder, and there is no power operator (^ is exclusive or). |
| lvl:arithmetic_operators@L0#2 \| level L0 way inconsistent_trace: Points to LT (evaluate operands, then apply the operator in the promoted type). Applied unevenly, a student uses integer division for one / and real division for another / in the same int expression. (here mixing in: "a^2 computes a squared") |
| lvl:arithmetic_operators@L1#0 \| level L1 way names_the_element: Arithmetic operators: the binary operators + - * / % that combine two numeric operands into a numeric result. |
| lvl:arithmetic_operators@L1#1 \| level L1 way states_the_rule: Both operands must be numeric, and the result has the promoted type (int, long or double). * / % bind tighter than + -, all group left to right, and int / or % by zero throws ArithmeticException. Unary minus negates its operand (-x) and unary plus leaves it unchanged; they bind more tightly than * and /, so -a * b is (-a) * b. |
| lvl:arithmetic_operators@L2#0 \| level L2 way line_by_line_trace: The left operand is evaluated fully, then the right operand, and then the operator is applied in the promoted type. |
| lvl:arithmetic_operators@L2#1 \| level L2 way output_prediction: With ints, 17 / 3 is 5 and 17 % 3 is 2. With doubles, 5.0 / 3.0 is 1.6666666666666667. With mixed operands, the int is promoted first (10 / 3.0 is 3.333...). |
| lvl:arithmetic_operators@L2#2 \| level L2 way one_step_execution: An arithmetic operator never changes its operands. Each variable is replaced by its value, and only an assignment (or ++ or op=) stores the result. |
| lvl:arithmetic_operators@L3#0 \| level L3 way relation_to_neighbour: Operators build larger expressions from smaller ones, and each operand can itself be any expression. |
| lvl:arithmetic_operators@L3#1 \| level L3 way relation_to_neighbour: For ints, / gives the truncated quotient and % the remainder, so (a / b) * b + a % b == a. |
| lvl:arithmetic_operators@L3#2 \| level L3 way relation_to_neighbour: int / or % by zero throws a run-time exception, while floating-point division by zero gives Infinity or NaN. |
| lvl:arithmetic_operators@L3#3 \| level L3 way relation_to_neighbour: + adds when both operands are numbers and concatenates when either operand is a String, converting the other operand to a String. It is left-associative, so 1 + 2 + "3" is "33" but "1" + 2 + 3 is "123". |
| lvl:arithmetic_operators@L3#4 \| level L3 way purpose_of_the_whole: Arithmetic operators turn formulas into code: totals, averages, unit conversions (inch * CM_PER_INCH), and digit extraction (n % 10, n / 10). |
| lvl:arithmetic_operators@L4#0 \| level L4 way predict_a_change: Changing an operand's type changes the operator's meaning (int / versus double /). Reordering * and / changes integer results (59 * 100 / 60 is 98, 59 / 60 * 100 is 0). Replacing ^ with Math.pow gives a real power. |
| lvl:arithmetic_operators@L4#1 \| level L4 way transfer_to_a_new_program: Use the operators for arithmetic on numeric types. Use the Math library for powers, roots and absolute values, and parse numeric Strings before computing with them. |
| lvl:arithmetic_operators@L4#2 \| level L4 way diagnose_an_error: rule broken: int / and % by zero are undefined. Kind: run-time exception. Trigger class: A divisor that evaluates to 0 (empty count, zero input): ArithmeticException: / by zero. |
| lvl:arithmetic_operators@L4#3 \| level L4 way diagnose_an_error: rule broken: Arithmetic operators other than + are not defined for Strings. Kind: compile-time. Trigger class: "10" - 1 or "6" * "2": 'bad operand types for binary operator'. |
| lvl:arithmetic_operators@L4#4 \| level L4 way diagnose_an_error: rule broken: ^ is not exponentiation. Kind: silent logic error. Trigger class: Writing x^2 or 10^6 for a power (10^6 is 12). |
| lvl:arithmetic_operators@L4#5 \| level L4 way diagnose_an_error: rule broken: int arithmetic wraps silently on overflow. Kind: silent logic error. Trigger class: Results outside the int range. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:arithmetic_operators@L2#1 (Score: 0.4)}
{lvl:arithmetic_operators@L4#0 (Score: 0.35)}
{lvl:arithmetic_operators@L2#0 (Score: 0.25)}
```
Kept by pruning: lvl:arithmetic_operators@L2#1 (0.40), lvl:arithmetic_operators@L4#0 (0.35), lvl:arithmetic_operators@L2#0 (0.25)

Label prompt for level way lvl:arithmetic_operators@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> With ints, 17 / 3 is 5 and 17 % 3 is 2. With doubles, 5.0 / 3.0 is 1.6666666666666667. With mixed operands, the int is promoted first (10 / 3.0 is 3.333...).

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.85
```

Label prompt for level way lvl:arithmetic_operators@L4#0 (the model saw only the concept, the way's name 'predict a change' and its text; no level):
> Changing an operand's type changes the operator's meaning (int / versus double /). Reordering * and / changes integer results (59 * 100 / 60 is 98, 59 / 60 * 100 is 0). Replacing ^ with Math.pow gives a real power.

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.6
```

Label prompt for level way lvl:arithmetic_operators@L2#0 (the model saw only the concept, the way's name 'line by line trace' and its text; no level):
> The left operand is evaluated fully, then the right operand, and then the operator is applied in the promoted type.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.85
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:arithmetic_operators@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:arithmetic_operators@L4#0 | shown | 0.35 | 0.6 | 0.21 | c2, c3 | The cast makes it a double division; so you get the decimal part |
| lvl:arithmetic_operators@L2#0 | not applied |  |  | 0.0 |  |  |

#### primitive_type, misconceptions: 2 edges offered
Edges offered:
| edge |
|---|
| mis:primitive_type->NewArrayHoldsGarbage \| wrong rule: The slots of a new int[n] hold unpredictable values until assigned. \| right rule: new fills every element with a default: 0 for numeric types, false for boolean, null for String and other non-primitive types. |
| mis:primitive_type->StringIsPrimitive \| wrong rule: String is a primitive type like int. \| right rule: String is not a primitive type; it is a class whose values are objects, although Java gives it some built-in support (literals, +). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:primitive_type->StringIsPrimitive (0.50), mis:primitive_type->NewArrayHoldsGarbage (0.50)
```
Kept by pruning: mis:primitive_type->StringIsPrimitive (0.50), mis:primitive_type->NewArrayHoldsGarbage (0.50)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:primitive_type->StringIsPrimitive \| wrong rule: String is a primitive type like int. \| right rule: String is not a primitive type; it is a class whose values are objects, although Java gives it some built-in support (literals, +). |
| mis:primitive_type->NewArrayHoldsGarbage \| wrong rule: The slots of a new int[n] hold unpredictable values until assigned. \| right rule: new fills every element with a default: 0 for numeric types, false for boolean, null for String and other non-primitive types. |
Model's reply:
```
mis:primitive_type->StringIsPrimitive | not applied | clauses: none | confidence: 0.95
mis:primitive_type->NewArrayHoldsGarbage | not applied | clauses: none | confidence: 0.95
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:primitive_type->StringIsPrimitive | not applied |  |  | 0.0 |  |  |
| mis:primitive_type->NewArrayHoldsGarbage | not applied |  |  | 0.0 |  |  |

#### primitive_type, level ways: 21 edges offered
Edges offered:
| edge |
|---|
| lvl:primitive_type@L0#0 \| level L0 way everyday_reading: In English 'primitive' means crude or old-fashioned, which suggests these are a lesser type or that anything simple-looking (String) is primitive. In Java a primitive type is one of eight types predefined by the language and named by a keyword. It holds a plain value of fixed size, so it has no methods and no overflow checking. |
| lvl:primitive_type@L0#2 \| level L0 way inconsistent_trace: Points to LT (each operation is done in its type's fixed size). Applied unevenly, a student truncates and wraps int results on one line but treats another int intermediate as an unbounded real number, e.g. tracing 59 / 60 * 100 as 98. (here mixing in: "String is a primitive type like int") |
| lvl:primitive_type@L1#0 \| level L1 way names_the_element: Primitive type: one of Java's eight built-in data types (boolean, char, byte, short, int, long, float, double). Its values are simple values, not objects. |
| lvl:primitive_type@L1#1 \| level L1 way states_the_rule: byte, short, int and long are 8/16/32/64-bit signed two's-complement integers. char is 16-bit unsigned. float and double are IEEE 754 32-bit and 64-bit. boolean has exactly the values true and false. The other four primitive types are byte and short (smaller integers), long (64-bit integers) and float (32-bit floating point). |
| lvl:primitive_type@L2#0 \| level L2 way line_by_line_trace: In an expression over primitives, the left operand is evaluated, then the right one. Both are then promoted to a common type (at least int, or long or double if either operand has that type), and the operator is applied in that type. |
| lvl:primitive_type@L2#1 \| level L2 way output_prediction: The result is a primitive value whose type comes from promotion: byte, short, char and int combinations give int, any long operand gives long, and any double operand gives double (e.g. 'a' + 1 is int 98, 1 + 2L is long 3, 1 + 2.0 is double 3.0). |
| lvl:primitive_type@L2#2 \| level L2 way one_step_execution: Assigning to a primitive variable copies the value into it. After int b = a; a = 3;, b still holds the old value because nothing is shared (unlike array references). |
| lvl:primitive_type@L3#0 \| level L3 way relation_to_neighbour: How a literal is written fixes its primitive type (1234 int, 3.14 double, true boolean, 'c' char), and it must fit the primitive type of the variable it initializes. |
| lvl:primitive_type@L3#1 \| level L3 way relation_to_neighbour: Between numeric primitive types, widening (int to double) happens automatically, while narrowing (double to int) needs a cast that drops information. boolean converts to no numeric type. |
| lvl:primitive_type@L3#2 \| level L3 way relation_to_neighbour: Each primitive type is a data type: a fixed set of values plus the operators defined for them. |
| lvl:primitive_type@L3#3 \| level L3 way relation_to_neighbour: int is the 32-bit signed integral primitive. Arithmetic on byte, short and char is also carried out in int. |
| lvl:primitive_type@L3#4 \| level L3 way relation_to_neighbour: double is the 64-bit IEEE 754 primitive. Once a double meets an integral operand in a binary operator, the whole operation becomes floating point. |
| lvl:primitive_type@L3#5 \| level L3 way relation_to_neighbour: boolean is the primitive with only true and false. It does not mix with numeric primitives: it cannot be cast to them, and only booleans may be used as conditions. |
| lvl:primitive_type@L3#6 \| level L3 way relation_to_neighbour: char is an integral primitive (16-bit unsigned UTF-16 code unit), so a char takes part in arithmetic as its numeric code. |
| lvl:primitive_type@L3#7 \| level L3 way relation_to_neighbour: A variable of a primitive type stores the value itself in its own memory. |
| lvl:primitive_type@L3#8 \| level L3 way purpose_of_the_whole: Primitive types hold the single numbers, characters and truth values that every computation is built from. You pick the one that matches the quantity: int for counts, double for measurements, boolean for conditions, char for one character. |
| lvl:primitive_type@L4#0 \| level L4 way predict_a_change: Switching one primitive type for another changes range and precision. int to long removes overflow up to about 9.2e18. int to double keeps fractions but stops being exact for very large integers. Changing a type to boolean breaks every arithmetic use. |
| lvl:primitive_type@L4#1 \| level L4 way transfer_to_a_new_program: Use a primitive type for a single numeric, character or true/false value. Use String for text, arrays for collections, BigInteger for integers beyond long, and int cents rather than double when money must be exact. |
| lvl:primitive_type@L4#2 \| level L4 way diagnose_an_error: rule broken: A narrowing primitive conversion needs an explicit cast. Kind: compile-time. Trigger class: Storing a double or long value in an int (or an int in a char) without a cast: 'possible lossy conversion'. |
| lvl:primitive_type@L4#3 \| level L4 way diagnose_an_error: rule broken: Integer primitives have fixed ranges and integer operators do not report overflow. Kind: silent logic error. Trigger class: Any int or long result outside the type's range (large products, factorials): the stored value has wrapped around. |
| lvl:primitive_type@L4#4 \| level L4 way diagnose_an_error: rule broken: Primitive type names are lowercase keywords. Kind: compile-time. Trigger class: Writing Int, Double or Boolean where int, double or boolean was meant, or string for String: 'cannot find symbol' or a type mismatch. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:primitive_type@L0#2 (Score: 0.5)}
{lvl:primitive_type@L3#4 (Score: 0.3)}
{lvl:primitive_type@L2#0 (Score: 0.2)}
```
Kept by pruning: lvl:primitive_type@L0#2 (0.50), lvl:primitive_type@L3#4 (0.30), lvl:primitive_type@L2#0 (0.20)

Label prompt for level way lvl:primitive_type@L0#2 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Points to LT (each operation is done in its type's fixed size). Applied unevenly, a student truncates and wraps int results on one line but treats another int intermediate as an unbounded real number, e.g. tracing 59 / 60 * 100 as 98. (here mixing in: "String is a primitive type like int")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.6
```

Label prompt for level way lvl:primitive_type@L3#4 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> double is the 64-bit IEEE 754 primitive. Once a double meets an integral operand in a binary operator, the whole operation becomes floating point.

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.6
```

Label prompt for level way lvl:primitive_type@L2#0 (the model saw only the concept, the way's name 'line by line trace' and its text; no level):
> In an expression over primitives, the left operand is evaluated, then the right one. Both are then promoted to a common type (at least int, or long or double if either operand has that type), and the operator is applied in that type.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.7
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:primitive_type@L0#2 | not applied |  |  | 0.0 |  |  |
| lvl:primitive_type@L3#4 | shown | 0.3 | 0.6 | 0.18 | c2, c3 | The cast makes it a double division; so you get the decimal part |
| lvl:primitive_type@L2#0 | not applied |  |  | 0.0 |  |  |

**Kept on a path at this depth** (best N in tiers: held misconception / broken edge, then held error, then correct / confirmed / shown):

lvl:arithmetic_operators@L4#0 [shown, tier 2, 0.21], lvl:primitive_type@L3#4 [shown, tier 2, 0.18]

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | 3.5 | mis:explicit_cast->CastAfterIntDivision |
| c2 | The cast makes it a double division | mis:explicit_cast->CastAfterIntDivision |
| c3 | so you get the decimal part | mis:explicit_cast->CastAfterIntDivision |

**Graph check, not yet read (the walk may not stop while these remain):** pre:arithmetic_operators->expression

**Figure 6 (the graph after depth 3)**

![Figure 6](figure06.png)
Held set after the one-fault rules: mis:explicit_cast->CastAfterIntDivision, mis:type_conversion->TargetTyping; filed at the dependent concept: mis:integer_division->CastAfterIntDivision

Shadows (a held label on a prerequisite, on a generic concept, or at a concept where the graph attaches the misconception as context, resting on the words of a stronger label at the specific or primary concept, is one wrong sentence read twice; it marks no root cause and sets no level): mis:type_conversion->TargetTyping (shadow of mis:explicit_cast->CastAfterIntDivision), mis:integer_division->CastAfterIntDivision (shadow of mis:explicit_cast->CastAfterIntDivision)


## 4. Diagnosis (rules over the labelled graph, no model)
Root-cause walk: held on ['explicit_cast'], broken at [] (consequences: pre:arithmetic_operators->integer_division, pre:integer_division->explicit_cast, pre:primitive_type->type_conversion, pre:type_conversion->explicit_cast), prerequisites {'expression': ['arithmetic_operators'], 'integer_division': ['arithmetic_operators'], 'explicit_cast': ['integer_division', 'type_conversion'], 'type_conversion': ['primitive_type'], 'println': ['statement']}
- arithmetic_operators: L4 via predict_a_change "The cast makes it a double division; so you get the decimal part"
- explicit_cast: L0 via misconception "3.5; The cast makes it a double division; so you get the decimal part"
- integer_division: L0 via inconsistent_trace "3.5; The cast makes it a double division; so you get the decimal part"
- primitive_type: L3 via relation_to_neighbour "The cast makes it a double division; so you get the decimal part"
- type_conversion: L0 via everyday_reading "3.5; The cast makes it a double division; so you get the decimal part"

**Stop:** max depth at depth 3; prompts {'prune': 13, 'label': 23}, fresh model calls 0, cache hits 36

### Held misconceptions
| concept | misconception | kind | quote | score | shares_evidence_with |
|---|---|---|---|---|---|
| explicit_cast | CastAfterIntDivision | misconception | 3.5; The cast makes it a double division; so you get the decimal part | 0.665 |  |

### Shadowed labels (held in the reading, folded into the label they shadow; not faults of their own)
| concept | misconception | shadow_of | quote | score |
|---|---|---|---|---|
| integer_division | CastAfterIntDivision | mis:explicit_cast->CastAfterIntDivision | 3.5; The cast makes it a double division; so you get the decimal part | 0.57 |
| type_conversion | TargetTyping | mis:explicit_cast->CastAfterIntDivision | 3.5; The cast makes it a double division; so you get the decimal part | 0.64 |

### Broken concept edges
| edge | quote | score | consequence_of |
|---|---|---|---|
| pre:arithmetic_operators->integer_division | 3.5; The cast makes it a double division; so you get the decimal part | 0.9 | mis:explicit_cast->CastAfterIntDivision |
| pre:integer_division->explicit_cast | 3.5; The cast makes it a double division; so you get the decimal part | 0.76 | mis:explicit_cast->CastAfterIntDivision |
| pre:primitive_type->type_conversion | 3.5; The cast makes it a double division; so you get the decimal part | 0.7 | mis:explicit_cast->CastAfterIntDivision |
| pre:type_conversion->explicit_cast | 3.5; The cast makes it a double division; so you get the decimal part | 0.15 | mis:explicit_cast->CastAfterIntDivision |

### Root cause: explicit_cast / CastAfterIntDivision

### Level per concept
| concept | level | way | misconception | source | quote | rule | also | note |
|---|---|---|---|---|---|---|---|---|
| arithmetic_operators | L4 | predict_a_change |  | level_way | The cast makes it a double division; so you get the decimal part |  |  |  |
| explicit_cast | L0 | misconception | CastAfterIntDivision | level_mis_links | 3.5; The cast makes it a double division; so you get the decimal part | wrong rule: (double) (3 / 5) is 0.6. Right rule: The integer division 3 / 5 = 0 happens inside the parentheses before the cast, so the result is 0.0; write (double) 3 / 5 to get 0.6. |  |  |
| integer_division | L0 | inconsistent_trace |  | level_way | 3.5; The cast makes it a double division; so you get the decimal part |  |  |  |
| primitive_type | L3 | relation_to_neighbour |  | level_way | The cast makes it a double division; so you get the decimal part |  |  |  |
| type_conversion | L0 | everyday_reading |  | level_way | 3.5; The cast makes it a double division; so you get the decimal part |  |  |  |

### Overall level: L0

Clause coverage (final):
| clause | covered by |
|---|---|
| 3.5 | mis:explicit_cast->CastAfterIntDivision |
| The cast makes it a double division | mis:explicit_cast->CastAfterIntDivision |
| so you get the decimal part | mis:explicit_cast->CastAfterIntDivision |

**Figure 7 (final picture)**

![Figure 7](figure07.png)


## 5. Every labelled edge (the evidence list, kept or not)
| depth | source | edge | kind | label | prune_score | confidence | score | clauses | quote |
|---|---|---|---|---|---|---|---|---|---|
| 1 | explicit_cast | mis:explicit_cast->CastAfterIntDivision | misconception | held | 0.7000000000000001 | 0.95 | 0.665 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 1 | explicit_cast | lvl:explicit_cast@L0#0 | level | shown | 0.35 | 0.6 | 0.21 | [1, 2] | 3.5; The cast makes it a double division |
| 1 | integer_division | mis:integer_division->CastAfterIntDivision | misconception | held | 0.6 | 0.95 | 0.57 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 1 | integer_division | lvl:integer_division@L0#4 | level | shown | 0.6 | 0.85 | 0.51 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 1 | type_conversion | mis:type_conversion->TargetTyping | misconception | held | 0.8 | 0.8 | 0.64 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 1 | type_conversion | lvl:type_conversion@L0#0 | level | shown | 0.25 | 0.72 | 0.18 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 2 | explicit_cast | pre:integer_division->explicit_cast | prerequisite | broken | 0.8 | 0.95 | 0.76 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 2 | explicit_cast | pre:type_conversion->explicit_cast | prerequisite | broken | 0.2 | 0.75 | 0.15 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 2 | type_conversion | pre:primitive_type->type_conversion | prerequisite | broken | 1.0 | 0.7 | 0.7 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 2 | integer_division | pre:arithmetic_operators->integer_division | prerequisite | broken | 1.0 | 0.9 | 0.9 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 3 | arithmetic_operators | lvl:arithmetic_operators@L4#0 | level | shown | 0.35 | 0.6 | 0.21 | [2, 3] | The cast makes it a double division; so you get the decimal part |
| 3 | primitive_type | lvl:primitive_type@L3#4 | level | shown | 0.3 | 0.6 | 0.18 | [2, 3] | The cast makes it a double division; so you get the decimal part |


## 6. The paths (what the keep step kept, depth by depth)
| depth | source | edge | kind | label | score | clauses |
|---|---|---|---|---|---|---|
| 1 | explicit_cast | mis:explicit_cast->CastAfterIntDivision | misconception | held | 0.665 | [1, 2, 3] |
| 1 | type_conversion | mis:type_conversion->TargetTyping | misconception | held | 0.64 | [1, 2, 3] |
| 1 | integer_division | mis:integer_division->CastAfterIntDivision | misconception | held | 0.57 | [1, 2, 3] |
| 2 | integer_division | pre:arithmetic_operators->integer_division | prerequisite | broken | 0.9 | [1, 2, 3] |
| 2 | explicit_cast | pre:integer_division->explicit_cast | prerequisite | broken | 0.76 | [1, 2, 3] |
| 2 | type_conversion | pre:primitive_type->type_conversion | prerequisite | broken | 0.7 | [1, 2, 3] |
| 2 | explicit_cast | pre:type_conversion->explicit_cast | prerequisite | broken | 0.15 | [1, 2, 3] |
| 3 | arithmetic_operators | lvl:arithmetic_operators@L4#0 | level | shown | 0.21 | [2, 3] |
| 3 | primitive_type | lvl:primitive_type@L3#4 | level | shown | 0.18 | [2, 3] |


## 7. Raw traversal log
```
clauses: 3.5 | The cast makes it a double division | so you get the decimal part
start: BM25 of the answer against each tagged concept: explicit_cast 7.80 (hits 1), integer_division 6.80 (hits 1), type_conversion 3.01 (hits 1), println 2.85 (hits 1), arithmetic_operators 2.45 (hits 1)
start: chosen explicit_cast, integer_division, type_conversion
start concepts (3): explicit_cast, integer_division, type_conversion
--- depth 1: current concepts ['explicit_cast', 'integer_division', 'type_conversion']
  explicit_cast (misconceptions): 6 candidates -> kept mis:explicit_cast->CastAfterIntDivision (0.70), mis:explicit_cast->LaterDoubleLiteralFixesDivision (0.20), mis:explicit_cast->CastAppliesToWholeExpression (0.10)
    mis:explicit_cast->CastAfterIntDivision -> held (0.67) "3.5; The cast makes it a double division; so you get the decimal part"
    mis:explicit_cast->LaterDoubleLiteralFixesDivision -> not applied (0.00) ""
    mis:explicit_cast->CastAppliesToWholeExpression -> not applied (0.00) ""
  explicit_cast (level ways): 19 candidates -> kept lvl:explicit_cast@L2#1 (0.40), lvl:explicit_cast@L0#0 (0.35), lvl:explicit_cast@L2#0 (0.25)
    lvl:explicit_cast@L2#1 -> not applied (0.00) ""
    lvl:explicit_cast@L0#0 -> shown (0.21) "3.5; The cast makes it a double division"
    lvl:explicit_cast@L2#0 -> not applied (0.00) ""
  integer_division (misconceptions): 6 candidates -> kept mis:integer_division->CastAfterIntDivision (0.60), mis:integer_division->TargetTyping (0.25), mis:integer_division->IntegerDivisionGivesFraction (0.15)
    mis:integer_division->CastAfterIntDivision -> held (0.57) "3.5; The cast makes it a double division; so you get the decimal part"
    mis:integer_division->TargetTyping -> not applied (0.00) ""
    mis:integer_division->IntegerDivisionGivesFraction -> not applied (0.00) ""
  integer_division (level ways): 16 candidates -> kept lvl:integer_division@L0#4 (0.60), lvl:integer_division@L2#0 (0.25), lvl:integer_division@L4#0 (0.15)
    lvl:integer_division@L0#4 -> shown (0.51) "3.5; The cast makes it a double division; so you get the decimal part"
    lvl:integer_division@L2#0 -> not applied (0.00) ""
    lvl:integer_division@L4#0 -> not applied (0.00) ""
  type_conversion (misconceptions): 3 candidates -> kept mis:type_conversion->TargetTyping (0.80), mis:type_conversion->NarrowingAssignmentAllowed (0.10), mis:type_conversion->NoCastIfSameSize (0.10)
    mis:type_conversion->TargetTyping -> held (0.64) "3.5; The cast makes it a double division; so you get the decimal part"
    mis:type_conversion->NarrowingAssignmentAllowed -> not applied (0.00) ""
    mis:type_conversion->NoCastIfSameSize -> not applied (0.00) ""
  type_conversion (level ways): 18 candidates -> kept lvl:type_conversion@L4#3 (0.40), lvl:type_conversion@L4#0 (0.35), lvl:type_conversion@L0#0 (0.25)
    lvl:type_conversion@L4#3 -> not applied (0.00) ""
    lvl:type_conversion@L4#0 -> not applied (0.00) ""
    lvl:type_conversion@L0#0 -> shown (0.18) "3.5; The cast makes it a double division; so you get the decimal part"
  kept: mis:explicit_cast->CastAfterIntDivision [held, tier 0, 0.67], mis:type_conversion->TargetTyping [held, tier 0, 0.64], mis:integer_division->CastAfterIntDivision [held, tier 0, 0.57]
  clause coverage: c1 <- mis:explicit_cast->CastAfterIntDivision; c2 <- mis:explicit_cast->CastAfterIntDivision; c3 <- mis:explicit_cast->CastAfterIntDivision
  not yet read: pre:arithmetic_operators->integer_division, pre:integer_division->explicit_cast, pre:primitive_type->type_conversion, pre:type_conversion->explicit_cast
--- depth 2: current concepts ['explicit_cast', 'type_conversion', 'integer_division']
  explicit_cast (concept edges): 2 candidates -> kept pre:integer_division->explicit_cast (0.80), pre:type_conversion->explicit_cast (0.20)
    pre:integer_division->explicit_cast -> broken (0.76) "3.5; The cast makes it a double division; so you get the decimal part"
    pre:type_conversion->explicit_cast -> broken (0.15) "3.5; The cast makes it a double division; so you get the decimal part"
  type_conversion (concept edges): 1 candidates -> kept pre:primitive_type->type_conversion (1.00)
    pre:primitive_type->type_conversion -> broken (0.70) "3.5; The cast makes it a double division; so you get the decimal part"
  integer_division (concept edges): 1 candidates -> kept pre:arithmetic_operators->integer_division (1.00)
    pre:arithmetic_operators->integer_division -> broken (0.90) "3.5; The cast makes it a double division; so you get the decimal part"
  kept: pre:arithmetic_operators->integer_division [broken, tier 0, 0.90], pre:integer_division->explicit_cast [broken, tier 0, 0.76], pre:primitive_type->type_conversion [broken, tier 0, 0.70], pre:type_conversion->explicit_cast [broken, tier 0, 0.15]
  reached across pre:arithmetic_operators->integer_division, entered next: arithmetic_operators
  reached across pre:primitive_type->type_conversion, entered next: primitive_type
  clause coverage: c1 <- mis:explicit_cast->CastAfterIntDivision; c2 <- mis:explicit_cast->CastAfterIntDivision; c3 <- mis:explicit_cast->CastAfterIntDivision
  not yet read: own edges of arithmetic_operators, own edges of primitive_type
--- depth 3: current concepts ['arithmetic_operators', 'integer_division', 'explicit_cast', 'primitive_type', 'type_conversion']
  arithmetic_operators (misconceptions): 3 candidates -> kept 
  arithmetic_operators (level ways): 18 candidates -> kept lvl:arithmetic_operators@L2#1 (0.40), lvl:arithmetic_operators@L4#0 (0.35), lvl:arithmetic_operators@L2#0 (0.25)
    lvl:arithmetic_operators@L2#1 -> not applied (0.00) ""
    lvl:arithmetic_operators@L4#0 -> shown (0.21) "The cast makes it a double division; so you get the decimal part"
    lvl:arithmetic_operators@L2#0 -> not applied (0.00) ""
  primitive_type (misconceptions): 2 candidates -> kept mis:primitive_type->StringIsPrimitive (0.50), mis:primitive_type->NewArrayHoldsGarbage (0.50)
    mis:primitive_type->StringIsPrimitive -> not applied (0.00) ""
    mis:primitive_type->NewArrayHoldsGarbage -> not applied (0.00) ""
  primitive_type (level ways): 21 candidates -> kept lvl:primitive_type@L0#2 (0.50), lvl:primitive_type@L3#4 (0.30), lvl:primitive_type@L2#0 (0.20)
    lvl:primitive_type@L0#2 -> not applied (0.00) ""
    lvl:primitive_type@L3#4 -> shown (0.18) "The cast makes it a double division; so you get the decimal part"
    lvl:primitive_type@L2#0 -> not applied (0.00) ""
  kept: lvl:arithmetic_operators@L4#0 [shown, tier 2, 0.21], lvl:primitive_type@L3#4 [shown, tier 2, 0.18]
  clause coverage: c1 <- mis:explicit_cast->CastAfterIntDivision; c2 <- mis:explicit_cast->CastAfterIntDivision; c3 <- mis:explicit_cast->CastAfterIntDivision
  not yet read: pre:arithmetic_operators->expression
  held: mis:explicit_cast->CastAfterIntDivision, mis:type_conversion->TargetTyping; filed at the dependent concept: mis:integer_division->CastAfterIntDivision
  shadows: mis:type_conversion->TargetTyping (shadow of mis:explicit_cast->CastAfterIntDivision), mis:integer_division->CastAfterIntDivision (shadow of mis:explicit_cast->CastAfterIntDivision)
  root-cause walk: held on ['explicit_cast'], broken at [] (consequences: pre:arithmetic_operators->integer_division, pre:integer_division->explicit_cast, pre:primitive_type->type_conversion, pre:type_conversion->explicit_cast), prerequisites {'expression': ['arithmetic_operators'], 'integer_division': ['arithmetic_operators'], 'explicit_cast': ['integer_division', 'type_conversion'], 'type_conversion': ['primitive_type'], 'println': ['statement']}
  level arithmetic_operators: L4 via predict_a_change "The cast makes it a double division; so you get the decimal part"
  level explicit_cast: L0 via misconception "3.5; The cast makes it a double division; so you get the decimal part"
  level integer_division: L0 via inconsistent_trace "3.5; The cast makes it a double division; so you get the decimal part"
  level primitive_type: L3 via relation_to_neighbour "The cast makes it a double division; so you get the decimal part"
  level type_conversion: L0 via everyday_reading "3.5; The cast makes it a double division; so you get the decimal part"
```


## 8. Full diagnosis JSON
```json
{
 "scenario": "MY29",
 "settings": {
  "width": 3,
  "depth": 3,
  "prune_tools": "llm",
  "model": "claude-opus-5-5",
  "backend": "claude-cli"
 },
 "clauses": [
  "3.5",
  "The cast makes it a double division",
  "so you get the decimal part"
 ],
 "start_concepts": [
  "explicit_cast",
  "integer_division",
  "type_conversion"
 ],
 "held_misconceptions": [
  {
   "concept": "explicit_cast",
   "misconception": "CastAfterIntDivision",
   "kind": "misconception",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "score": 0.665
  }
 ],
 "error_evidence": [],
 "shadowed": [
  {
   "concept": "integer_division",
   "misconception": "CastAfterIntDivision",
   "shadow_of": "mis:explicit_cast->CastAfterIntDivision",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "score": 0.57
  },
  {
   "concept": "type_conversion",
   "misconception": "TargetTyping",
   "shadow_of": "mis:explicit_cast->CastAfterIntDivision",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "score": 0.64
  }
 ],
 "broken_edges": [
  {
   "edge": "pre:arithmetic_operators->integer_division",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "score": 0.9,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "edge": "pre:integer_division->explicit_cast",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "score": 0.76,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "edge": "pre:primitive_type->type_conversion",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "score": 0.7,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "edge": "pre:type_conversion->explicit_cast",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "score": 0.15,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  }
 ],
 "root_cause": {
  "concept": "explicit_cast",
  "misconception": "CastAfterIntDivision"
 },
 "levels": {
  "arithmetic_operators": {
   "level": "L4",
   "way": "predict_a_change",
   "quote": "The cast makes it a double division; so you get the decimal part",
   "edge": "lvl:arithmetic_operators@L4#0",
   "source": "level_way",
   "also_shown": []
  },
  "explicit_cast": {
   "level": "L0",
   "way": "misconception",
   "misconception": "CastAfterIntDivision",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "edge": "mis:explicit_cast->CastAfterIntDivision",
   "rule": "wrong rule: (double) (3 / 5) is 0.6. Right rule: The integer division 3 / 5 = 0 happens inside the parentheses before the cast, so the result is 0.0; write (double) 3 / 5 to get 0.6.",
   "source": "level_mis_links"
  },
  "integer_division": {
   "level": "L0",
   "way": "inconsistent_trace",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "edge": "lvl:integer_division@L0#4",
   "source": "level_way",
   "also_shown": []
  },
  "primitive_type": {
   "level": "L3",
   "way": "relation_to_neighbour",
   "quote": "The cast makes it a double division; so you get the decimal part",
   "edge": "lvl:primitive_type@L3#4",
   "source": "level_way",
   "also_shown": []
  },
  "type_conversion": {
   "level": "L0",
   "way": "everyday_reading",
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "edge": "lvl:type_conversion@L0#0",
   "source": "level_way",
   "also_shown": []
  }
 },
 "overall_level": "L0",
 "paths": [
  {
   "depth": 1,
   "source": "explicit_cast",
   "edge": "mis:explicit_cast->CastAfterIntDivision",
   "target": "CastAfterIntDivision",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.95,
   "prune_score": 0.7000000000000001,
   "score": 0.665,
   "logic": "(double) (3 / 5) is 0.6.",
   "twin_logic": "The integer division 3 / 5 = 0 happens before the cast, so the result is 0.0.",
   "role": "primary"
  },
  {
   "depth": 1,
   "source": "type_conversion",
   "edge": "mis:type_conversion->TargetTyping",
   "target": "TargetTyping",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.8,
   "prune_score": 0.8,
   "score": 0.64,
   "logic": "The type of a numerical expression depends on the type expected by the surrounding context, so double d = a / b; keeps the fraction.",
   "twin_logic": "The type of a numerical expression is determined by the expression, so double d = a / b; with int a and b still divides as ints.",
   "role": "context"
  },
  {
   "depth": 1,
   "source": "integer_division",
   "edge": "mis:integer_division->CastAfterIntDivision",
   "target": "CastAfterIntDivision",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.95,
   "prune_score": 0.6,
   "score": 0.57,
   "logic": "(double) (3 / 5) is 0.6.",
   "twin_logic": "The integer division 3 / 5 = 0 happens before the cast, so the result is 0.0.",
   "role": "context"
  },
  {
   "depth": 2,
   "source": "integer_division",
   "edge": "pre:arithmetic_operators->integer_division",
   "target": "arithmetic_operators",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.9,
   "prune_score": 1.0,
   "score": 0.9,
   "logic": "For ints a and b, / gives the whole quotient q and % gives the remainder r, so that a = b * q + r.",
   "twin_logic": null,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "depth": 2,
   "source": "explicit_cast",
   "edge": "pre:integer_division->explicit_cast",
   "target": "integer_division",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.95,
   "prune_score": 0.8,
   "score": 0.76,
   "logic": "To divide two ints and keep the fraction, one operand is cast to double before the division, as in (double) freq[i] / T; casting the finished quotient comes too late.",
   "twin_logic": null,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "depth": 2,
   "source": "type_conversion",
   "edge": "pre:primitive_type->type_conversion",
   "target": "primitive_type",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.7,
   "prune_score": 1.0,
   "score": 0.7,
   "logic": "Conversion moves a value between primitive types: int to double happens automatically, while double to int needs a cast that drops the fraction.",
   "twin_logic": null,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "depth": 2,
   "source": "explicit_cast",
   "edge": "pre:type_conversion->explicit_cast",
   "target": "type_conversion",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.75,
   "prune_score": 0.2,
   "score": 0.15,
   "logic": "A cast is the explicit form of type conversion between primitive types, required when the conversion may lose information (e.g. (int) 2.71828 is 2).",
   "twin_logic": null,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "depth": 3,
   "source": "arithmetic_operators",
   "edge": "lvl:arithmetic_operators@L4#0",
   "target": "arithmetic_operators@L4",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.6,
   "prune_score": 0.35,
   "score": 0.21,
   "logic": "Changing an operand's type changes the operator's meaning (int / versus double /). Reordering * and / changes integer results (59 * 100 / 60 is 98, 59 / 60 * 100 is 0). Replacing ^ with Math.pow gives a real power.",
   "twin_logic": null,
   "level": "L4",
   "way": "predict_a_change"
  },
  {
   "depth": 3,
   "source": "primitive_type",
   "edge": "lvl:primitive_type@L3#4",
   "target": "primitive_type@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.6,
   "prune_score": 0.3,
   "score": 0.18,
   "logic": "double is the 64-bit IEEE 754 primitive. Once a double meets an integral operand in a binary operator, the whole operation becomes floating point.",
   "twin_logic": null,
   "level": "L3",
   "way": "relation_to_neighbour"
  }
 ],
 "evidence": [
  {
   "depth": 1,
   "source": "explicit_cast",
   "edge": "mis:explicit_cast->CastAfterIntDivision",
   "target": "CastAfterIntDivision",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.95,
   "prune_score": 0.7000000000000001,
   "score": 0.665,
   "logic": "(double) (3 / 5) is 0.6.",
   "twin_logic": "The integer division 3 / 5 = 0 happens before the cast, so the result is 0.0.",
   "role": "primary"
  },
  {
   "depth": 1,
   "source": "explicit_cast",
   "edge": "lvl:explicit_cast@L0#0",
   "target": "explicit_cast@L0",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    2
   ],
   "quote": "3.5; The cast makes it a double division",
   "confidence": 0.6,
   "prune_score": 0.35,
   "score": 0.21,
   "logic": "Folk meaning: 'convert 2.7 to a whole number' means round it to 3, and (int) looks like a function label that applies to everything written after it. Java rule: a cast to int truncates toward zero ((int) 2.7 is 2) and, having higher precedence than arithmetic, applies only to the value immediately following it.",
   "twin_logic": null,
   "level": "L0",
   "way": "everyday_reading"
  },
  {
   "depth": 1,
   "source": "integer_division",
   "edge": "mis:integer_division->CastAfterIntDivision",
   "target": "CastAfterIntDivision",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.95,
   "prune_score": 0.6,
   "score": 0.57,
   "logic": "(double) (3 / 5) is 0.6.",
   "twin_logic": "The integer division 3 / 5 = 0 happens before the cast, so the result is 0.0.",
   "role": "context"
  },
  {
   "depth": 1,
   "source": "integer_division",
   "edge": "lvl:integer_division@L0#4",
   "target": "integer_division@L0",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.85,
   "prune_score": 0.6,
   "score": 0.51,
   "logic": "Points to LT (each / looks at its own two operand types when it is evaluated). Applied unevenly, a student evaluates 9/5 as 1.8 in f = (9/5)*c + 32 but as 1 elsewhere, or assumes a later double operand rescues an earlier int division. (here mixing in: \"The type of a numerical expression depends on the type expected by the surrounding context, so double d = a / b; keeps the fraction\")",
   "twin_logic": null,
   "level": "L0",
   "way": "inconsistent_trace"
  },
  {
   "depth": 1,
   "source": "type_conversion",
   "edge": "mis:type_conversion->TargetTyping",
   "target": "TargetTyping",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.8,
   "prune_score": 0.8,
   "score": 0.64,
   "logic": "The type of a numerical expression depends on the type expected by the surrounding context, so double d = a / b; keeps the fraction.",
   "twin_logic": "The type of a numerical expression is determined by the expression, so double d = a / b; with int a and b still divides as ints.",
   "role": "context"
  },
  {
   "depth": 1,
   "source": "type_conversion",
   "edge": "lvl:type_conversion@L0#0",
   "target": "type_conversion@L0",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.72,
   "prune_score": 0.25,
   "score": 0.18,
   "logic": "Folk meaning: a number is a number, so 3, 3.0 and \"3\" are the same value, and a variable will take whatever form the context needs (the result 'becomes' double when stored in a double). Java rule: every value has a fixed type; conversions happen only by automatic promotion to a wider type, by an explicit cast, by conversion methods (Integer.parseInt, Math.round), or by String conversion with +.",
   "twin_logic": null,
   "level": "L0",
   "way": "everyday_reading"
  },
  {
   "depth": 2,
   "source": "explicit_cast",
   "edge": "pre:integer_division->explicit_cast",
   "target": "integer_division",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.95,
   "prune_score": 0.8,
   "score": 0.76,
   "logic": "To divide two ints and keep the fraction, one operand is cast to double before the division, as in (double) freq[i] / T; casting the finished quotient comes too late.",
   "twin_logic": null,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "depth": 2,
   "source": "explicit_cast",
   "edge": "pre:type_conversion->explicit_cast",
   "target": "type_conversion",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.75,
   "prune_score": 0.2,
   "score": 0.15,
   "logic": "A cast is the explicit form of type conversion between primitive types, required when the conversion may lose information (e.g. (int) 2.71828 is 2).",
   "twin_logic": null,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "depth": 2,
   "source": "type_conversion",
   "edge": "pre:primitive_type->type_conversion",
   "target": "primitive_type",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.7,
   "prune_score": 1.0,
   "score": 0.7,
   "logic": "Conversion moves a value between primitive types: int to double happens automatically, while double to int needs a cast that drops the fraction.",
   "twin_logic": null,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "depth": 2,
   "source": "integer_division",
   "edge": "pre:arithmetic_operators->integer_division",
   "target": "arithmetic_operators",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.9,
   "prune_score": 1.0,
   "score": 0.9,
   "logic": "For ints a and b, / gives the whole quotient q and % gives the remainder r, so that a = b * q + r.",
   "twin_logic": null,
   "consequence_of": "mis:explicit_cast->CastAfterIntDivision"
  },
  {
   "depth": 3,
   "source": "arithmetic_operators",
   "edge": "lvl:arithmetic_operators@L4#0",
   "target": "arithmetic_operators@L4",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.6,
   "prune_score": 0.35,
   "score": 0.21,
   "logic": "Changing an operand's type changes the operator's meaning (int / versus double /). Reordering * and / changes integer results (59 * 100 / 60 is 98, 59 / 60 * 100 is 0). Replacing ^ with Math.pow gives a real power.",
   "twin_logic": null,
   "level": "L4",
   "way": "predict_a_change"
  },
  {
   "depth": 3,
   "source": "primitive_type",
   "edge": "lvl:primitive_type@L3#4",
   "target": "primitive_type@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.6,
   "prune_score": 0.3,
   "score": 0.18,
   "logic": "double is the 64-bit IEEE 754 primitive. Once a double meets an integral operand in a binary operator, the whole operation becomes floating point.",
   "twin_logic": null,
   "level": "L3",
   "way": "relation_to_neighbour"
  }
 ],
 "clause_coverage": {
  "3.5": "mis:explicit_cast->CastAfterIntDivision",
  "The cast makes it a double division": "mis:explicit_cast->CastAfterIntDivision",
  "so you get the decimal part": "mis:explicit_cast->CastAfterIntDivision"
 },
 "unexplained_clauses": [],
 "stop_reason": "max depth",
 "depth_reached": 3,
 "llm_calls": 0,
 "llm_cache_hits": 36,
 "llm_calls_by_step": {
  "prune": 13,
  "label": 23
 }
}
```
